In [1]:
from datasets import load_dataset

dataset = load_dataset("pameydorke/arcanumsearch-embs")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/384 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 19.0MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1153 [00:00<?, ? examples/s]

In [2]:
import numpy as np

def parse_fast(example):
    arr = np.fromstring(example["embedding"].strip("[]"), sep=", ", dtype=np.float32)
    example["embedding"] = arr.tolist()
    return example

dataset = dataset.map(parse_fast)

Map:   0%|          | 0/1153 [00:00<?, ? examples/s]

In [3]:
# from sentence_transformers import (
#     SentenceTransformer,
# )

# model = SentenceTransformer('pameydorke/arcanum-cross-platform-retriever')

In [4]:
import torch

embeddings = torch.tensor(dataset["train"]["embedding"], dtype=torch.float32)

In [5]:
import torch.nn.functional as F

def retrieve(query, top_k=5):
    query_embs = model.encode(query, convert_to_tensor=True)

    similarities = F.cosine_similarity(query_embs, embeddings)

    scores, indices = torch.topk(similarities, k=top_k)

    return [
        {"score": float(scores[i]), "url": dataset["train"][indices[i]]["url"], "text": dataset["train"][indices[i]]["content"]}
        for i in range(top_k)
    ]

In [6]:
# TODO: recall, ndcg

In [7]:
sample_dataset = dataset["train"].shuffle(seed=42).select(range(100))

In [17]:
i = 17

sample_dataset[i]["url"]

'https://arcanum.fandom.com/wiki/Character_background'

In [20]:
j = 13

sample_dataset[j]["url"]

'https://arcanum.fandom.com/wiki/Character_background'

In [24]:
dataset

DatasetDict({
    train: Dataset({
        features: ['content', 'url', 'embedding', 'title'],
        num_rows: 1153
    })
})

In [ ]:
dataset["train"].to_pandas().drop_duplicates('url')

,content,url,embedding,title
0,Pyrotechnic Bow. Pyrotechnic Bow. The Pyrotech...,https://arcanum.fandom.com/wiki/Pyrotechnic_Bow,"[0.002190711908042431, 0.0052022249437868595, ...",Pyrotechnic Bow
1,Original Post: What aspects of Arcanum do you ...,https://reddit.com/r/arcanum/comments/1w2qsg/w...,"[0.04089789837598801, -0.026058191433548927, -...",What aspects of Arcanum do you wish other game...
2,"Tulla. Tulla. ""I can't STAND Tulla..everyone i...",https://arcanum.fandom.com/wiki/Tulla,"[-0.008103189058601856, 0.04381492733955383, 0...",Tulla
3,Succour Beast. Succour Beast. Succour Beast is...,https://arcanum.fandom.com/wiki/Succour_Beast,"[-0.054103221744298935, 0.04578685760498047, -...",Succour Beast
4,"Gar. Gar. ""He has the body of an Orc, but the ...",https://arcanum.fandom.com/wiki/Gar,"[-0.025474006310105324, 0.003153689904138446, ...",Gar
...,...,...,...,...
1147,Tell Lianna Pel Dar about Maximillian. Tell Li...,https://arcanum.fandom.com/wiki/Tell_Lianna_Pe...,"[-0.039290592074394226, 0.004452085122466087, ...",Tell Lianna Pel Dar about Maximillian
1149,Original Post: Can't Mod Random Encounter. I h...,https://reddit.com/r/arcanum/comments/wn68gh/c...,"[0.007532842922955751, -0.0052304393611848354,...",Can't Mod Random Encounter
1150,Looking Glass Rifle. Looking Glass Rifle. This...,https://arcanum.fandom.com/wiki/Looking_Glass_...,"[-0.04116610437631607, -0.016336340457201004, ...",Looking Glass Rifle
1151,Side Arms and Saltpeter. Side Arms and Saltpet...,https://arcanum.fandom.com/wiki/Side_Arms_and_...,"[0.015276065096259117, 0.027656786143779755, -...",Side Arms and Saltpeter
